In [ ]:
import psutil
import sys
# !{sys.executable} --version
# !{sys.executable} -m pip install shap --upgrade 
import joblib
import time
from collections import defaultdict
from shutil import copy
import numpy as np
import pandas as pd
import random
import os
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from matplotlib import cm

from tqdm.notebook import tqdm
import seaborn as sns
from collections import Counter

from glob import glob
import psi4
from helper_CC_ML_spacial import *

from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF
from sklearn.kernel_ridge import KernelRidge
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.neighbors import KNeighborsRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.decomposition import PCA
from sklearn.metrics import root_mean_squared_error, r2_score, mean_absolute_error
from sklearn.pipeline import Pipeline

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import LogNorm
from matplotlib import cm
from matplotlib.colors import Normalize
from matplotlib.ticker import FuncFormatter
from matplotlib import font_manager
from matplotlib.ticker import ScalarFormatter
from matplotlib.ticker import LogFormatterSciNotation
font_path = 'Futura Book.ttf'
font_manager.fontManager.addfont(font_path)
prop = font_manager.FontProperties(fname=font_path, size='large')
plt.rcParams['font.family'] = prop.get_name()
plt.rcParams.update({'font.size': 12})

In [ ]:
def formatter(x, pos):
    if x != 0:
        return f'{int(x/1000)}k'
    else:
        return 0

In [ ]:
# Most important features (top 5 by SHAP):
# doublecheck: Numerator of the MP2 t2-amplitude, two-electron integral <ik || ab>
# t2start: Initial MP2 t2-amplitude
# t2mag: Magnitude of the MP2 t2-amplitude
# orbdiff: Denominator of the MP2 t2-amplitude
# diag: Binary feature denoting whether a=b (virtual orbits are the same)
top5 = ['doublecheck', 't2start', 't2mag', 'orbdiff', 'diag']

# 31 Features order in X, including t2
properties=['Evir1', 'Hvir1', 'Jvir1', 'Kvir1', 'Evir2', 'Hvir2', 'Jvir2', 'Kvir2', 'Eocc1', 'Jocc1', 'Kocc1', 'Hocc1','Eocc2', 'Jocc2', 'Kocc2', 'Hocc2', 'Jia1', 'Jia2', 'Kia1', 'Kia2','diag', 'orbdiff', 'doublecheck', 't2start', 't2mag', 't2sign', 'Jia1mag', 'Jia2mag','Kia1mag', 'Kia2mag','t2']

In [ ]:
basis_sets = ['STO-3G', 'cc-pVDZ', 'aug-cc-pVDZ']

In [ ]:
molecules = ["water", "methanol", "ethylene", "ethane", "methane", "ammonia", "formaldehyde"]

In [ ]:
random.seed(0)
all_sampled_files = []

all_files = sorted(glob(os.path.join("data", f"water*.xyz")))
sample_size = min(150, len(all_files))
sampled_files = random.sample(all_files, sample_size)
train = sampled_files[:100] # 100 train molecules
test = sampled_files[100:] # 50 test molecules

In [ ]:
train

In [ ]:
test

In [ ]:
import xgboost as xgb

In [ ]:
from sklearn.neighbors import KNeighborsRegressor

In [ ]:
#########################################

In [ ]:
filenames = test.copy()
filenames[:5]

In [ ]:
import joblib

basis = "STO-3G"
n = 100

fn = f"water_out/knn_{basis}_model_{n}.pkl"
model = joblib.load(fn)

In [ ]:
model

In [ ]:
#basis_sets = ['STO-3G']

filenames = test.copy()

n_list = [10, 20, 40, 60, 80, 100] # iterating over N=10, 20, 40, 60, 80, 100 training molecules

mean_deviations = []
mean_iter_to_converge = []

for basis in basis_sets:
    for n in n_list:
        print(f"Processing N={n} molecules with {basis} basis set")
        
        fn = f"water_out/knn_{basis}_model_{n}.pkl"
        model = joblib.load(fn)
        
        ml_iter_list = []
        ml_dev = []
        ml_time_to_converge_list = []
        for fn in filenames:
            print(f"Processing {fn}")       # import xyz file
            with open(fn,'r') as f:
                text=f.read()
            mol = psi4.geometry(text)                
            
            psi4.core.clean()
            psi4.core.be_quiet()
            
            psi4.set_options({'basis': basis,
                              'scf_type':     'pk',
                              'reference':    'rohf',
                              'mp2_type':     'conv',
                              'e_convergence': 1e-8,
                              'd_convergence': 1e-8})

            rhf_e, scf_wfn = psi4.energy('scf', return_wfn=True)        # rhf_e is Hartree-Fock energy 
            scf_e, scf_wfn = psi4.energy('scf', return_wfn=True)
            
            A=HelperCCEnergy(mol, rhf_e, scf_wfn,freeze_core=True)

            MP2T2=A.t2start
            A.t1 = np.zeros((A.t1.shape))
            A.t2 = MP2T2
            
            MP2E = A.compute_energy(iterate=False)                    # MP2 (initial) energy
            CCSDE = A.compute_energy() 
            

            A=HelperCCEnergy(mol, rhf_e, scf_wfn,freeze_core=True)
            
            # Here we are initializing with the ML predicted t2-amplitudes, first we predict using the top 5 features
            y_pred = model.predict(np.vstack([getattr(A,i).flatten() for i in top5]).T)
            
            A.t1 = np.zeros((A.t1.shape))
            A.t2 = y_pred.reshape(*A.t2.shape)
            A.t2start = y_pred.reshape(*A.t2.shape)

            mlE_0 = A.compute_energy(iterate=False)   # ML-predicted energy
            #mlE = A.compute_energy()                # exact CCSD energy

            
            ml_dev.append(abs(mlE_0-CCSDE))  # energy deviation from CCSD if we directly initialize with ML-predicted amplitudes


            print("Deviation here: ", abs(mlE_0-CCSDE), "\n")


            # Now for 
            mlE = A.compute_energy() # iterate using ML-predicted t2 as aninitial guess
            
            mlHistory = A.history               # tuple, (iteration_nunber, CCSDcorr_E)
            mlHistory.insert(0, (0,mlE_0))     # inserting the INITIAL t2 amplitude from ML method
            
            #for i in mlHistory:
            #    print(i)
            #print("\n")
            
            iterations = len(mlHistory)     # number of iterations to converge
            #print(f"No. of iterations to converge: {iterations}")
            ml_iter_list.append(iterations)
            
            ml_time_to_converge = A.time_to_converge 
            ml_time_to_converge_list.append(ml_time_to_converge)
            #print(f"Time to converge: {ml_time_to_converge}")
            
            features = pd.DataFrame(np.array([getattr(A,attr).flatten() for attr in top5]).T,columns=top5)
            #print(f"Features: \n{features}")

        print("\n")
        print("Iterations for each molecule list: ", ml_iter_list)
        print("Mean iterations to converge: ", np.mean(ml_iter_list))
        print("Deviations from CCSD for each molecule: ", ml_dev)
        print(f"Mean deviation from CCSD: {np.mean(ml_dev)} (Eₕ)")
        print("Times to converge for each molecule: ", ml_time_to_converge_list)
        print("Mean time to converge", np.mean(ml_time_to_converge_list), " seconds")

        mean_deviations.append((basis, n, np.mean(ml_dev)))
        mean_iter_to_converge.append((basis, n, np.mean(ml_iter_list)))
        print("aa: here: ", mean_deviations)
        print("Next model...\n")


In [ ]:
grouped = defaultdict(lambda: ([], []))  # {basis: ([n], [dev])}

for basis, n, dev in mean_deviations:
    grouped[basis][0].append(n)
    grouped[basis][1].append(dev * 1000)  # Convert E_h to mE_h

# Plot
fig, ax = plt.subplots(figsize=(8, 6))

marker_styles = {
    'STO-3G': 'x',      # cross
    'cc-pVDZ': '^',     # triangle
    'aug-cc-pVDZ': '+'  # plus
}

color_styles = {
    'STO-3G': 'darkblue',
    'cc-pVDZ': 'darkgreen',
    'aug-cc-pVDZ': 'darkred'
}

for basis, (n_list, dev_list) in grouped.items():
    marker = marker_styles.get(basis, 'x')
    color = color_styles.get(basis, 'black')

    if marker == 'x':
        ax.plot(n_list, dev_list, linestyle='-', marker=marker, label=basis,
                markersize=7, markeredgewidth=4, color=color)  # Thicker cross
    else:
        ax.plot(n_list, dev_list, linestyle='-', marker=marker, label=basis,
                markersize=7, markeredgewidth=2, color=color)

ax.set_xlabel("Training molecules", fontsize=15)
ax.set_ylabel("Deviation from CCSD (m$E_h$)", fontsize=15, labelpad=14)

ax.grid(True, which="both", linestyle="--", linewidth=0.5)
ax.tick_params(axis='both', labelsize=13)
ax.legend(fontsize=13)

plt.tight_layout()
plt.xlim(0, 110)
#plt.savefig("figs/ccsd_energy.png", dpi=300)
plt.show()

In [ ]:
grouped = defaultdict(lambda: ([], []))  # {basis: ([n], [dev])}

for basis, n, it in mean_iter_to_converge:
    grouped[basis][0].append(n)
    grouped[basis][1].append(it)  # Convert E_h to mE_h

# Plot
fig, ax = plt.subplots(figsize=(8, 6))

marker_styles = {
    'STO-3G': 'x',      # cross
    'cc-pVDZ': '^',     # triangle
    'aug-cc-pVDZ': '+'  # plus
}

color_styles = {
    'STO-3G': 'darkblue',
    'cc-pVDZ': 'darkgreen',
    'aug-cc-pVDZ': 'darkred'
}

for basis, (n_list, dev_list) in grouped.items():
    marker = marker_styles.get(basis, 'x')
    color = color_styles.get(basis, 'black')

    if marker == 'x':
        ax.plot(n_list, dev_list, linestyle='-', marker=marker, label=basis,
                markersize=7, markeredgewidth=4, color=color)  # Thicker cross
    else:
        ax.plot(n_list, dev_list, linestyle='-', marker=marker, label=basis,
                markersize=7, markeredgewidth=2, color=color)

ax.set_xlabel("Training molecules", fontsize=15)
ax.set_ylabel("Iterations to convergence", fontsize=15, labelpad=14)

ax.grid(True, which="both", linestyle="--", linewidth=0.5)
ax.tick_params(axis='both', labelsize=13)
ax.legend(fontsize=13)

plt.tight_layout()
plt.xlim(0, 110)
#plt.savefig("figs/iterations.png", dpi=300)
plt.show()

In [ ]:
grouped = defaultdict(lambda: ([], []))  
for basis, n, dev in mean_deviations:
    grouped[basis][0].append(n)
    grouped[basis][1].append(dev * 1000)  

fig, ax = plt.subplots(figsize=(8, 6))
#ax.set_aspect(0.6)

marker_styles = {'STO-3G': 'x', 'cc-pVDZ': '^', 'aug-cc-pVDZ': '+'}
color_styles = {'STO-3G': 'darkblue', 'cc-pVDZ': 'darkgreen', 'aug-cc-pVDZ': 'darkred'}

for basis, (n_list, dev_list) in grouped.items():
    marker = marker_styles.get(basis, 'x')
    color = color_styles.get(basis, 'black')
    ax.plot(n_list, dev_list, linestyle='-', marker=marker, label=basis,
            markersize=7, markeredgewidth=4 if marker=='x' else 2, color=color)

ax.set_xlabel("Training molecules", fontsize=15)
ax.set_ylabel("Deviation from CCSD (m$E_h$)", fontsize=15, labelpad=14)
ax.grid(True, which="both", linestyle="--", linewidth=0.5)
ax.tick_params(axis='both', labelsize=13)
ax.legend(fontsize=13)
ax.set_xlim(0, 110)
plt.tight_layout()
#plt.savefig("figs/ccsd_energy.png", dpi=300)
plt.show()

grouped = defaultdict(lambda: ([], []))
for basis, n, it in mean_iter_to_converge:
    grouped[basis][0].append(n)
    grouped[basis][1].append(it)

fig, ax = plt.subplots(figsize=(8, 6))
#ax.set_aspect(0.6)

for basis, (n_list, dev_list) in grouped.items():
    marker = marker_styles.get(basis, 'x')
    color = color_styles.get(basis, 'black')
    ax.plot(n_list, dev_list, linestyle='-', marker=marker, label=basis,
            markersize=7, markeredgewidth=4 if marker=='x' else 2, color=color)

ax.set_xlabel("Training molecules", fontsize=15)
ax.set_ylabel("Iterations to convergence", fontsize=15, labelpad=14)
ax.grid(True, which="both", linestyle="--", linewidth=0.5)
ax.tick_params(axis='both', labelsize=13)
ax.legend(fontsize=13)
ax.set_xlim(0, 110)
plt.tight_layout()
#plt.savefig("figs/iterations.png", dpi=300)
plt.show()

In [ ]:

fake_data = [
    np.random.normal(loc=0, scale=1, size=100),   # STO-3g
    np.random.normal(loc=1, scale=1.5, size=100), # cc-pVDZ
    np.random.normal(loc=-1, scale=0.5, size=100) # aug-cc-pVDZ
]

fig, ax = plt.subplots(figsize=(5, 4))

ax.boxplot(
    fake_data,
    labels=["STO-3G", "cc-pVDZ", "aug-cc-pVDZ"],
    patch_artist=True,
    boxprops=dict(facecolor="white", color="black"),
    medianprops=dict(color="black"),
    whiskerprops=dict(color="black"),
    capprops=dict(color="black"),
    flierprops=dict(markerfacecolor="white", marker="o", markersize=5, linestyle="none", markeredgecolor="black")
)

plt.ylabel("Energy deviation (m$E_h$)", labelpad=4)
plt.title("Fake data for now...")
#plt.grid(axis="y", linestyle="--", alpha=0.7)
#plt.savefig("figs/boxplot.png", dpi=300)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

molecules = ['water', 'methanol', 'ethylene', 'ethane', 'methane', 'ammonia', 'formaldehyde']
# water, methanol, ethylene, ethane, methane, ammonia, and formaldehyde

values1 = [5, 7, 3, 8, 6, 4, 7]  # STO-3G
values2 = [6, 5, 4, 7, 5, 6, 8]  # cc-pVDZ
values3 = [4, 6, 5, 6, 7, 5, 6]  # aug-cc-pVDZ

x = np.arange(len(molecules))
width = 0.25  # width of each bar

plt.bar(x - width, values1, width, label='STO-3G', alpha=0.7, edgecolor='black')
plt.bar(x, values2, width, label='cc-pVDZ', alpha=0.7, edgecolor='black')
plt.bar(x + width, values3, width, label='aug-cc-pVDZ', alpha=0.7, edgecolor='black')

plt.xticks(x, molecules, fontsize=15)
plt.yticks(fontsize=15)

plt.ylabel('Energy deviation (m$E_h$)', labelpad=8, fontsize=15)

plt.legend(fontsize=13)
plt.title("Fake data for now...")
#plt.savefig("figs/barchart.png", dpi=300)
plt.show()

In [ ]:
plt.hist(y_test, bins=30, edgecolor='black')
plt.xlabel("Values of y_test")
plt.ylabel("Frequency")
plt.title("Distribution of y_test")
plt.show()

In [ ]:
plt.hist(y_pred, bins=30, edgecolor='black')
plt.xlabel("Values of y_pred")
plt.ylabel("Frequency")
plt.title("Distribution of y_pred")
plt.show()

# Feature analysis

In [ ]:
################################################################################

In [ ]:
basis = "STO-3G"
n = 100
data = joblib.load(f"water_out/{basis}_train_data_splits_{n}.pkl")

X_train = data["X_train"]
y_train = data["y_train"]

data = joblib.load(f"water_out/{basis}_test_data_splits.pkl")
X_test = data["X_test"]
y_test = data["y_test"]

In [ ]:
# Clustering plots for the dataset
from sklearn.pipeline import make_pipeline
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import umap

In [ ]:
# Dimensionality reduction using PCA
t1 = time.time()
pca = PCA(n_components=2)
X_train_pca = pca.fit_transform(X_train)
X_test_pca = pca.transform(X_test)
t2 = time.time()
print(f"Time taken: {t2-t1} seconds")

In [ ]:
# Create the scatter plot with a continuous color range using "Spectral" palette
plt.figure(figsize=(7, 5))
scatter_train = sns.scatterplot(x=X_train_pca[:, 0], y=X_train_pca[:, 1], hue=y_train,  palette="viridis", alpha=0.5)
scatter_test = sns.scatterplot(x=X_test_pca[:, 0], y=X_test_pca[:, 1], hue=y_test, palette="viridis", marker='X')
plt.xlabel('PCA Component 1')
plt.ylabel('PCA Component 2')
plt.title('Dimensionality reduction using PCA')
#plt.savefig("figs/pca.svg", format='svg', dpi=300)
plt.show()

In [ ]:
# Dimensionality reduction using t-SNE
tsne = TSNE(n_components=2)
X_train_tsne = tsne.fit_transform(X_train)
X_test_tsne = tsne.fit_transform(X_test)

plt.figure(figsize=(7, 5))
sns.scatterplot(x=X_train_tsne[:, 0], y=X_train_tsne[:, 1], label='Train', alpha=0.6)
sns.scatterplot(x=X_test_tsne[:, 0], y=X_test_tsne[:, 1], label='Test', alpha=0.6, marker='X')
plt.xlabel('t-SNE Component 1')
plt.ylabel('t-SNE Component 2')
plt.title('Dimensionality reduction using t-SNE')
plt.legend()
#plt.savefig("figs/tsne.svg", format='svg', dpi=300)
plt.show()

# Dimensionality reduction using UMAP
reducer = umap.UMAP()
X_train_umap = reducer.fit_transform(X_train)
X_test_umap = reducer.transform(X_test)

plt.figure(figsize=(7, 5))
sns.scatterplot(x=X_train_umap[:, 0], y=X_train_umap[:, 1], label='Train', alpha=0.6)
sns.scatterplot(x=X_test_umap[:, 0], y=X_test_umap[:, 1], label='Test', alpha=0.6, marker='X')
plt.xlabel('UMAP Component 1')
plt.ylabel('UMAP Component 2')
plt.title('Dimensionality reduction using UMAP')
plt.legend()

#plt.savefig("figs/umap.svg", format='svg', dpi=300)
plt.show()

In [ ]:
X = np.concatenate([X_train, X_test], axis=0)
X_df = pd.DataFrame(X, columns=top5)

In [ ]:
X_df

In [ ]:
y_full = np.concatenate([y_train, y_test])
X_df['t2'] = y_full

In [ ]:
cols = ['doublecheck', 't2start', 't2mag', 'orbdiff', 'diag', 't2']

In [ ]:
# Correlation matrix
corr = X_df.corr()

# Plot heatmap correctly
fig, ax = plt.subplots(figsize=(8, 6))
cax = ax.matshow(corr, cmap="Blues")

# Add colorbar
fig.colorbar(cax)

# Set ticks with feature names
ax.set_xticks(range(len(cols)))
ax.set_xticklabels(cols, rotation=90, fontsize=10, weight='bold')
ax.set_yticks(range(len(cols)))
ax.set_yticklabels(cols, fontsize=10, weight='bold')

#plt.savefig("figs/corr_top5.svg", format='svg', dpi=300)
plt.show()

In [ ]:
plt.rcParams['axes.unicode_minus'] = False
# Correlation matrix
corr = X_df.corr()

# Plot heatmap correctly
fig, ax = plt.subplots(figsize=(8, 6))
cax = ax.matshow(corr, cmap="Blues", vmin=-1, vmax=1)  # include -1 for even range

# Add colorbar with label
cbar = fig.colorbar(cax)
cbar.set_label("Correlation coefficient", fontsize=12)

# Add feature labels
ax.set_xticks(range(len(cols)))
ax.set_xticklabels(cols, rotation=90, fontsize=10, weight='bold')
ax.set_yticks(range(len(cols)))
ax.set_yticklabels(cols, fontsize=10, weight='bold')

# Move x-axis labels to bottom
ax.xaxis.set_ticks_position('bottom')

# Add axis labels
ax.set_xlabel("Features", fontsize=12, weight='bold')
ax.set_ylabel("Features", fontsize=12, weight='bold')

# Add values in each square
#for (i, j), val in np.ndenumerate(corr.values):
#    ax.text(j, i, f"{val:.2f}", ha='center', va='center', color='black', fontsize=8)

for (i, j), val in np.ndenumerate(corr.values):
    if val < 0:
        display_val = f"-- {abs(val):.2f}"
    else:
        display_val = f"{val:.2f}"
    ax.text(j, i, display_val, ha='center', va='center', color='black', fontsize=10, weight='bold')


plt.tight_layout()
#plt.savefig("figs/corr_top5.png", dpi=300)
plt.show()


In [ ]:
plot_df = X_df.copy()
target = 't2'

n_features = plot_df.shape[1] - 1  # exclude target
n_cols = 7
n_rows = (n_features - 1) // n_cols + 1

fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 4, n_rows * 3))
axes = axes.flatten()  # Flatten to simplify indexing

for i, col in enumerate(plot_df.columns):
    if col == target:
        continue
    sns.scatterplot(
        x=plot_df[col],
        y=plot_df[target],
        ax=axes[i],
        color="xkcd:blue with a hint of purple",
        legend=False
    )
    axes[i].set_xlabel(col, fontweight='bold')
    axes[i].xaxis.set_label_position('top')
    axes[i].set_ylabel(target if i % n_cols == 0 else '')

for j in range(len(plot_df.columns)-1, len(axes)):
    fig.delaxes(axes[j])

fig.tight_layout()

#plt.savefig("figs/corr_per_feature.svg", format='svg', dpi=300)
plt.show()

In [ ]:
basis = "STO-3G"
n = 100
splits_fn = f"/mnt/c/Users/Maxim/Documents/Toronto/Research/code/DDLUCJ/data/{basis}_train_data_splits_{n}.pkl"
data = joblib.load(splits_fn)

X_train = data["X_train"]
y_train = data["y_train"]

splits_fn = f"/mnt/c/Users/Maxim/Documents/Toronto/Research/code/DDLUCJ/data/{basis}_test_data_splits.pkl"
data = joblib.load(splits_fn)
X_test = data["X_test"]
y_test = data["y_test"]

In [ ]:
# SHAP feature importances
import shap
best_model = joblib.load('out/optimised_STO-3G_best_model_100.pkl')
tree_model = best_model.named_steps['xgb']

In [ ]:
top5 = ['doublecheck', 't2start', 't2mag', 'orbdiff', 'diag']

In [ ]:
explainer = shap.TreeExplainer(tree_model)
shap_values = explainer.shap_values(X_train)
shap.summary_plot(shap_values, X_train, feature_names=top5)

In [ ]:
plt.figure(figsize=(18,10))

shap.summary_plot(
    shap_values, 
    X_train, 
    feature_names=top5,  
    plot_type='bar', 
    show=False
)

#plt.xticks(fontsize=10) 
#plt.yticks(fontsize=12)
plt.xlabel("mean(|SHAP value|)", fontsize=12, labelpad=10)
plt.ylabel("Top 5 Features", fontsize=14, labelpad=40)

for patch in plt.gca().patches:
    patch.set_facecolor("skyblue") 

plt.show()

In [ ]:
tree_model.feature_importances_        # XGBoost feature importance values match the SHAP analysis

In [ ]:
test_shap_values = explainer.shap_values(X_test)
shap.summary_plot(test_shap_values, X_test, feature_names=top5)

In [ ]:
plt.figure(figsize=(18,10))

shap.summary_plot(
    shap_values, 
    X_train, 
    feature_names=top5,  
    plot_type='bar', 
    show=False
)

#plt.xticks(fontsize=10) 
#plt.yticks(fontsize=12)
plt.xlabel("mean(|SHAP value|)", fontsize=12, labelpad=10)
plt.ylabel("Top 5 Features", fontsize=14, labelpad=40)

for patch in plt.gca().patches:
    patch.set_facecolor("skyblue") 

##plt.savefig("figs/shap_analysis.svg", format='svg', dpi=300)
plt.show()

In [ ]:
plt.figure(figsize=(18, 10))

shap.summary_plot(
    shap_values, 
    X_train, 
    feature_names=top5,  
    plot_type='bar', 
    show=False
)

plt.xlabel("mean(|SHAP value|)", fontsize=12, labelpad=10)
plt.ylabel("Features", fontsize=14, labelpad=20)

# Set bar color and add value at the end of each bar
for patch in plt.gca().patches:
    patch.set_facecolor("skyblue")
    width = patch.get_width()
    plt.text(
        width + 0.0001,  # slightly after the end of the bar
        patch.get_y() + patch.get_height() / 2,
        f"{width:.3f}",  # show value with 3 decimals
        va='center',
        fontsize=10,
        weight='bold'
    )

plt.tight_layout()
#plt.savefig("figs/shap_analysis.png", dpi=300)
plt.show()


In [ ]:
#####

# Scaling of Random Forest

In [ ]:
basis = "STO-3G"

data = joblib.load(f"water_out/{basis}_test_data_splits.pkl")
X_test = data["X_test"]
y_test = data["y_test"]

In [ ]:
n_list = [10, 20, 40, 60, 80, 100]
for n in n_list:
    print(f"Train size: {n}")
    data = joblib.load(f"water_out/{basis}_train_data_splits_{n}.pkl")
    
    X_train = data["X_train"]
    y_train = data["y_train"]
    
         
    model = RandomForestRegressor()
        
    model_pipeline = Pipeline([
        ('scaler', MinMaxScaler(feature_range=(-1,1))),
        ('regressor', model)
    ])
        
    # fit model
    t1 = time.time()
    model_pipeline.fit(X_train, y_train)
    t2 = time.time()
    y_pred = model_pipeline.predict(X_test)
    
    # compute performance metrics
    r2 = r2_score(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    rmse = root_mean_squared_error(y_test, y_pred)

    with open("water_out/deletemodel.pkl", "wb") as f:
        joblib.dump(model, f)

    file_size = os.path.getsize("water_out/deletemodel.pkl")
    print(f"Model size: {file_size / 1e6:.2f} MB")
    
    print(f"N: {n}, MAE: {mae}, RMSE: {rmse}, R2: {r2}, Training time {t2-t1}\n")